In [ ]:
!pwd
!ls /root/.cache/kagglehub/datasets/bwandowando/dota-2-pro-league-matches-2023/versions/257/2016
!echo ""
!head -n3 /root/.cache/kagglehub/datasets/bwandowando/dota-2-pro-league-matches-2023/versions/257/2016/main_metadata.csv

/content
all_word_counts.csv  main_metadata.csv	radiant_exp_adv.csv
chat.csv	     objectives.csv	radiant_gold_adv.csv
cosmetics.csv	     picks_bans.csv	teamfights.csv
draft_timings.csv    players.csv	teams.csv

"","match_id","barracks_status_dire","barracks_status_radiant","cluster","dire_score","duration","engine","first_blood_time","game_mode","human_players","leagueid","lobby_type","match_seq_num","negative_votes","positive_votes","radiant_score","radiant_win","start_date_time","tower_status_dire","tower_status_radiant","version","replay_salt","series_id","series_type","patch","region","throw","loss","comeback","stomp","replay_url","dire_team_id","radiant_team_id"
0,2049344492,"'00111111'","'00111111'",133,0,2153,1,192,2,10,4194,1,1804607126,63.0,107.0,0,True,"2016-01-02 15:12:19","'0000011100100000'","'0000011110111110'","",1224511497.0,37073.0,1.0,16,3.0,1017.0,16389.0,"","","http://replay133.valve.net/570/2049344492_1224511497.dem.bz2","",""
1,2049487960,"'00111111'","'00111111'"

In [1]:
import kagglehub
import pandas as pd
import os
import zipfile

# 1. Загрузка датасета
print("Идет загрузка датасета (около 100-200 МБ)...")
path = kagglehub.dataset_download("bwandowando/dota-2-pro-league-matches-2023")
print("Путь к файлам:", path)

# 2. Распаковка (если скачался архив)
data_path = path
if path.endswith('.zip'):
    print("Обнаружен архив, распаковываем...")
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall('/content/kaggle_dataset')
    data_path = '/content/kaggle_dataset'

print("Содержимое папки:", os.listdir(data_path))


Идет загрузка датасета (около 100-200 МБ)...


100%|██████████| 8.60G/8.60G [07:47<00:00, 19.8MB/s]

Extracting files...


Путь к файлам: /root/.cache/kagglehub/datasets/bwandowando/dota-2-pro-league-matches-2023/versions/257
Содержимое папки: ['202604', '202601', '202606', '202607', '202603', '2025', '202609', '2016', '202608', 'Constants', '2020', '2017', '2018', '2022', '2021', '202602', 'Images', '2019', '2024', '2023', '202605']


In [ ]:
path = "/root/.cache/kagglehub/datasets/bwandowando/dota-2-pro-league-matches-2023/versions/257"

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error

# --- 0. ПУТИ ---
base = path
year_dir = os.path.join(base, '2023')

# --- 1. ЗАГРУЗКА ---
print("Загрузка main_metadata.csv...")
df_match = pd.read_csv(os.path.join(year_dir, 'main_metadata.csv'))

print("Загрузка players.csv...")
df_players = pd.read_csv(os.path.join(year_dir, 'players.csv'))

print("match:", df_match.shape, "| players:", df_players.shape)

# --- 2. СБОРКА ДРАФТА ---
# player_slot: 0-4 Radiant, 128-132 Dire
# Группируем по match_id и собираем список героев для каждой команды

radiant_heroes = (
    df_players[df_players['player_slot'] < 5]
    .sort_values(['match_id', 'player_slot'])
    .groupby('match_id')['hero_id']
    .apply(list)
    .to_dict()
)

dire_heroes = (
    df_players[df_players['player_slot'] >= 128]
    .sort_values(['match_id', 'player_slot'])
    .groupby('match_id')['hero_id']
    .apply(list)
    .to_dict()
)

# Добавляем колонки r1..r5 и d1..d5
# Берём только матчи, где есть ровно 5 героев с каждой стороны
valid_matches = set(radiant_heroes) & set(dire_heroes)
valid_matches = {m for m in valid_matches if len(radiant_heroes[m]) == 5 and len(dire_heroes[m]) == 5}

df_match = df_match[df_match['match_id'].isin(valid_matches)].copy()

for i in range(5):
    df_match[f'r{i+1}_hero'] = df_match['match_id'].map(lambda mid: radiant_heroes[mid][i])
    df_match[f'd{i+1}_hero'] = df_match['match_id'].map(lambda mid: dire_heroes[mid][i])

# --- 3. ПОДГОТОВКА ДАННЫХ ---
X_class = df_match[[f'r{i}_hero' for i in range(1, 6)] + [f'd{i}_hero' for i in range(1, 6)]]
y_class = df_match['radiant_win']

# Регрессия — из players.csv
# В players.csv уже есть колонка 'kda', но пересчитаем для прозрачности
df_players_reg = df_players[df_players['leaver_status'] == 0].copy()
df_players_reg['kda_calc'] = (df_players_reg['kills'] + df_players_reg['assists']) / (df_players_reg['deaths'] + 1)
regression_features = ['gold_per_min', 'xp_per_min', 'hero_damage', 'tower_damage', 'level']
X_reg = df_players_reg[regression_features]
y_reg = df_players_reg['kda_calc']

print("Классификация: X", X_class.shape, "| y", y_class.shape)
print("Регрессия:     X", X_reg.shape, "| y", y_reg.shape)

Загрузка main_metadata.csv...
Загрузка players.csv...


/tmp/ipykernel_790/4191250381.py:18: DtypeWarning: Columns (67,70,71,124,143) have mixed types. Specify dtype option on import or set low_memory=False.
  df_players = pd.read_csv(os.path.join(year_dir, 'players.csv'))


match: (29810, 43) | players: (298031, 149)
Классификация: X (29802, 10) | y (29802,)
Регрессия:     X (271814, 5) | y (271814,)


In [3]:
# ============================================================
# ЛАБОРАТОРНАЯ №2. ПУНКТ 2 — БЕЙЗЛАЙН (логистическая и линейная регрессия)
# ============================================================

from sklearn.linear_model import LogisticRegression, LinearRegression

print("=" * 60)
print("ЛАБ. №2: БЕЙЗЛАЙН — ЛОГИСТИЧЕСКАЯ И ЛИНЕЙНАЯ РЕГРЕССИЯ")
print("=" * 60)

# --- Классификация: логистическая регрессия ---
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

log_clf = LogisticRegression(max_iter=1000, random_state=42)
log_clf.fit(X_train_c_scaled, y_train_c)
y_pred_c = log_clf.predict(X_test_c_scaled)
y_proba_c = log_clf.predict_proba(X_test_c_scaled)[:, 1]

print("\n--- Классификация: LogisticRegression (бейзлайн) ---")
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_c):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test_c, y_proba_c):.3f}")
print(f"F1:       {f1_score(y_test_c, y_pred_c):.3f}")
print("(KNN из лаб. №1: Acc 0.499, AUC 0.496, F1 0.504)")

# --- Регрессия: линейная регрессия ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

lin_reg = LinearRegression()
lin_reg.fit(X_train_r_scaled, y_train_r)
y_pred_r = lin_reg.predict(X_test_r_scaled)

print("\n--- Регрессия: LinearRegression (бейзлайн) ---")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.3f}")
print(f"MAE:  {mean_absolute_error(y_test_r, y_pred_r):.3f}")
print("(KNN из лаб. №1: RMSE 3.895, MAE 2.304)")

ЛАБ. №2: БЕЙЗЛАЙН — ЛОГИСТИЧЕСКАЯ И ЛИНЕЙНАЯ РЕГРЕССИЯ

--- Классификация: LogisticRegression (бейзлайн) ---
Accuracy: 0.505
ROC-AUC:  0.506
F1:       0.560
(KNN из лаб. №1: Acc 0.499, AUC 0.496, F1 0.504)

--- Регрессия: LinearRegression (бейзлайн) ---
RMSE: 3.826
MAE:  2.355
(KNN из лаб. №1: RMSE 3.895, MAE 2.304)


In [4]:
# ============================================================
# ЛАБ. №2. ПУНКТ 3 — УЛУЧШЕНИЕ БЕЙЗЛАЙНА
# ============================================================

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge

baseline_lr = {
    'clf': {'Accuracy': 0.505, 'ROC-AUC': 0.506, 'F1': 0.560},
    'reg': {'RMSE': 3.826, 'MAE': 2.355},
}

# ---------- ГИПОТЕЗА 1: One-hot encoding (классификация) ----------
print("=" * 60)
print("ГИПОТЕЗА 1: One-hot encoding героев")
print("=" * 60)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_class_ohe = ohe.fit_transform(X_class)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class_ohe, y_class, test_size=0.2, random_state=42, stratify=y_class
)

log_clf_ohe = LogisticRegression(max_iter=2000, random_state=42)
log_clf_ohe.fit(X_train_c, y_train_c)
y_pred = log_clf_ohe.predict(X_test_c)
y_proba = log_clf_ohe.predict_proba(X_test_c)[:, 1]

print(f"Accuracy: {accuracy_score(y_test_c, y_pred):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test_c, y_proba):.3f}")
print(f"F1:       {f1_score(y_test_c, y_pred):.3f}")
print("(бейзлайн: Acc 0.505, AUC 0.506, F1 0.560)")

# ---------- ГИПОТЕЗА 2: Подбор C на кросс-валидации (классификация) ----------
print("\n" + "=" * 60)
print("ГИПОТЕЗА 2: Подбор C (регуляризация) на кросс-валидации")
print("=" * 60)

param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=42),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1
)
grid.fit(X_train_c, y_train_c)

print("Лучший C:", grid.best_params_)
print(f"Лучший CV ROC-AUC: {grid.best_score_:.3f}")

best_clf = grid.best_estimator_
y_pred = best_clf.predict(X_test_c)
y_proba = best_clf.predict_proba(X_test_c)[:, 1]
print(f"Test Accuracy: {accuracy_score(y_test_c, y_pred):.3f}")
print(f"Test ROC-AUC:  {roc_auc_score(y_test_c, y_proba):.3f}")
print(f"Test F1:       {f1_score(y_test_c, y_pred):.3f}")

# ---------- ГИПОТЕЗА 3: class_weight='balanced' (классификация) ----------
print("\n" + "=" * 60)
print("ГИПОТЕЗА 3: class_weight='balanced'")
print("=" * 60)
print("Распределение классов:", y_class.value_counts().to_dict())

log_clf_bal = LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced')
log_clf_bal.fit(X_train_c, y_train_c)
y_pred = log_clf_bal.predict(X_test_c)
y_proba = log_clf_bal.predict_proba(X_test_c)[:, 1]

print(f"Accuracy: {accuracy_score(y_test_c, y_pred):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test_c, y_proba):.3f}")
print(f"F1:       {f1_score(y_test_c, y_pred):.3f}")

# ---------- ГИПОТЕЗА 4: Удаление выбросов KDA (регрессия) ----------
print("\n" + "=" * 60)
print("ГИПОТЕЗА 4: Удаление выбросов KDA (топ-1%)")
print("=" * 60)

threshold = y_reg.quantile(0.99)
print(f"Порог 99-го перцентиля KDA: {threshold:.2f}")
print(f"Выбросов: {(y_reg > threshold).sum()} из {len(y_reg)}")

mask = y_reg <= threshold
X_reg_clean = X_reg[mask]
y_reg_clean = y_reg[mask]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg_clean, y_reg_clean, test_size=0.2, random_state=42
)
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

lin_reg_clean = LinearRegression()
lin_reg_clean.fit(X_train_r_scaled, y_train_r)
y_pred = lin_reg_clean.predict(X_test_r_scaled)

print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred)):.3f}")
print(f"MAE:  {mean_absolute_error(y_test_r, y_pred):.3f}")
print("(бейзлайн: RMSE 3.826, MAE 2.355)")

# ---------- ГИПОТЕЗА 5: Ridge-регрессия (регрессия) ----------
print("\n" + "=" * 60)
print("ГИПОТЕЗА 5: Ridge-регрессия (мультиколлинеарность)")
print("=" * 60)

# Проверим корреляцию признаков
print("Корреляция признаков:")
print(X_reg_clean.corr().round(2))

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_r_scaled, y_train_r)
y_pred = ridge.predict(X_test_r_scaled)

print(f"Ridge RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred)):.3f}")
print(f"Ridge MAE:  {mean_absolute_error(y_test_r, y_pred):.3f}")

ГИПОТЕЗА 1: One-hot encoding героев
Accuracy: 0.529
ROC-AUC:  0.539
F1:       0.542
(бейзлайн: Acc 0.505, AUC 0.506, F1 0.560)

ГИПОТЕЗА 2: Подбор C (регуляризация) на кросс-валидации
Лучший C: {'C': 0.01}
Лучший CV ROC-AUC: 0.541
Test Accuracy: 0.527
Test ROC-AUC:  0.547
Test F1:       0.555

ГИПОТЕЗА 3: class_weight='balanced'
Распределение классов: {True: 15042, False: 14760}
Accuracy: 0.527
ROC-AUC:  0.539
F1:       0.534

ГИПОТЕЗА 4: Удаление выбросов KDA (топ-1%)
Порог 99-го перцентиля KDA: 23.00
Выбросов: 2686 из 271814
RMSE: 3.450
MAE:  2.084
(бейзлайн: RMSE 3.826, MAE 2.355)

ГИПОТЕЗА 5: Ridge-регрессия (мультиколлинеарность)
Корреляция признаков:
              gold_per_min  xp_per_min  hero_damage  tower_damage  level
gold_per_min          1.00        0.83         0.63          0.65   0.70
xp_per_min            0.83        1.00         0.61          0.49   0.79
hero_damage           0.63        0.61         1.00          0.43   0.73
tower_damage          0.65        0.49     

In [5]:
# ============================================================
# ЛАБ. №2. ИТОГОВЫЙ УЛУЧШЕННЫЙ БЕЙЗЛАЙН
# ============================================================

print("=" * 60)
print("ЛАБ. №2: ИТОГОВЫЙ УЛУЧШЕННЫЙ БЕЙЗЛАЙН — КЛАССИФИКАЦИЯ")
print("=" * 60)

ohe_final = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_class_ohe_final = ohe_final.fit_transform(X_class)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class_ohe_final, y_class, test_size=0.2, random_state=42, stratify=y_class
)

final_clf = LogisticRegression(C=0.01, max_iter=2000, random_state=42)
final_clf.fit(X_train_c, y_train_c)
y_pred = final_clf.predict(X_test_c)
y_proba = final_clf.predict_proba(X_test_c)[:, 1]

print(f"Accuracy: {accuracy_score(y_test_c, y_pred):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test_c, y_proba):.3f}")
print(f"F1:       {f1_score(y_test_c, y_pred):.3f}")
print("(бейзлайн: Acc 0.505, AUC 0.506, F1 0.560)")

print("\n" + "=" * 60)
print("ЛАБ. №2: ИТОГОВЫЙ УЛУЧШЕННЫЙ БЕЙЗЛАЙН — РЕГРЕССИЯ")
print("=" * 60)

threshold = y_reg.quantile(0.99)
mask = y_reg <= threshold
X_reg_clean = X_reg[mask]
y_reg_clean = y_reg[mask]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg_clean, y_reg_clean, test_size=0.2, random_state=42
)
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

final_reg = LinearRegression()
final_reg.fit(X_train_r_scaled, y_train_r)
y_pred = final_reg.predict(X_test_r_scaled)

print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred)):.3f}")
print(f"MAE:  {mean_absolute_error(y_test_r, y_pred):.3f}")
print("(бейзлайн: RMSE 3.826, MAE 2.355)")

ЛАБ. №2: ИТОГОВЫЙ УЛУЧШЕННЫЙ БЕЙЗЛАЙН — КЛАССИФИКАЦИЯ
Accuracy: 0.527
ROC-AUC:  0.547
F1:       0.555
(бейзлайн: Acc 0.505, AUC 0.506, F1 0.560)

ЛАБ. №2: ИТОГОВЫЙ УЛУЧШЕННЫЙ БЕЙЗЛАЙН — РЕГРЕССИЯ
RMSE: 3.450
MAE:  2.084
(бейзлайн: RMSE 3.826, MAE 2.355)


## 5. Имплементация линейной и логистической регрессии (с нуля)

По заданию (п. 4) реализуем собственные классы на numpy — без `sklearn.linear_model`.

**`MyLinearRegression`:**
- Обучение через нормальное уравнение `w = (X^T X)^(-1) X^T y` (с `np.linalg.lstsq` для устойчивости при мультиколлинеарности).
- `predict` — матричное умножение `X @ w`.

**`MyLogisticRegression`:**
- Обучение через градиентный спуск: `w := w - lr * X^T (sigmoid(Xw) - y) / n`.
- L2-регуляризация (аналог `C` в sklearn): `w := w - lr * (X^T (sigmoid(Xw) - y) / n + alpha * w)`.
- `predict` — порог 0.5, `predict_proba` — sigmoid.

**Зачем:** сравнить свою реализацию со sklearn — убедиться, что результаты совпадают.

In [15]:
import numpy as np

class MyLinearRegression:
    """Линейная регрессия через нормальное уравнение (с intercept)."""
    def __init__(self):
        self.w = None
        self.intercept_ = None
        self.coef_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        # добавляем столбец единиц для свободного члена
        X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
        w_aug, *_ = np.linalg.lstsq(X_aug, y, rcond=None)
        self.intercept_ = w_aug[0]
        self.coef_ = w_aug[1:]
        self.w = w_aug
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
        return X_aug @ self.w


class MyLogisticRegression:
    def __init__(self, lr=0.1, n_iters=1000, alpha=0.0):
        self.lr = lr
        self.n_iters = n_iters
        self.alpha = alpha
        self.w = None

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64)
        n, d = X.shape
        self.w = np.zeros(d, dtype=np.float64)

        for it in range(self.n_iters):
            z = X @ self.w
            p = self._sigmoid(z)
            grad = X.T @ (p - y) / n + self.alpha * self.w
            self.w -= self.lr * grad
            # защита от разлёта
            if not np.all(np.isfinite(self.w)):
                print(f"Расходимость на итерации {it}, alpha={self.alpha}, lr={self.lr}")
                self.w = np.zeros(d)  # сброс
                break
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float64)
        return self._sigmoid(X @ self.w)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

## 6. Сравнение MyLinear/MyLogistic со sklearn на бейзлайне (п. 4a–4e)

Прогоняем свои реализации на тех же данных и с теми же параметрами, что sklearn в бейзлайне (п. 2): сырые `hero_id` для классификации, 5 числовых признаков для регрессии, `StandardScaler`.

Для `MyLogisticRegression` используем подвыборку 15k строк — градиентный спуск на 30k × 1240 (one-hot) был бы медленным, а для проверки корректности подвыборки достаточно.

In [16]:
print("=" * 60)
print("MyLinear/MyLogistic НА БЕЙЗЛАЙНЕ (п. 4a–4e)")
print("=" * 60)

# --- Классификация: LogReg на сырых hero_id ---
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)
scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

# sklearn — для сверки
sk_clf = LogisticRegression(max_iter=1000, random_state=42)
sk_clf.fit(X_train_c_scaled, y_train_c)
sk_pred = sk_clf.predict(X_test_c_scaled)
sk_proba = sk_clf.predict_proba(X_test_c_scaled)[:, 1]

# MyLogisticRegression — на подвыборке train (для скорости)
n_sub = min(15000, X_train_c_scaled.shape[0])
idx_sub = np.random.RandomState(42).choice(X_train_c_scaled.shape[0], n_sub, replace=False)
X_train_sub = X_train_c_scaled[idx_sub]
y_train_sub = y_train_c.values[idx_sub]

my_clf = MyLogisticRegression(lr=0.1, n_iters=1000, alpha=0.0)
my_clf.fit(X_train_sub, y_train_sub)
my_pred = my_clf.predict(X_test_c_scaled)
my_proba = my_clf.predict_proba(X_test_c_scaled)

print("\n--- Классификация: MyLogisticRegression vs sklearn (бейзлайн) ---")
print(f"MyLogReg  Accuracy: {accuracy_score(y_test_c, my_pred):.3f} | "
      f"ROC-AUC: {roc_auc_score(y_test_c, my_proba):.3f} | "
      f"F1: {f1_score(y_test_c, my_pred):.3f}")
print(f"sklearn   Accuracy: {accuracy_score(y_test_c, sk_pred):.3f} | "
      f"ROC-AUC: {roc_auc_score(y_test_c, sk_proba):.3f} | "
      f"F1: {f1_score(y_test_c, sk_pred):.3f}")

# --- Регрессия: MyLinearRegression ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

# sklearn
sk_reg = LinearRegression()
sk_reg.fit(X_train_r_scaled, y_train_r)
sk_pred_r = sk_reg.predict(X_test_r_scaled)

# MyLinearRegression
my_reg = MyLinearRegression()
my_reg.fit(X_train_r_scaled, y_train_r.values)
my_pred_r = my_reg.predict(X_test_r_scaled)

print("\n--- Регрессия: MyLinearRegression vs sklearn (бейзлайн) ---")
print(f"MyLinReg  RMSE: {np.sqrt(mean_squared_error(y_test_r, my_pred_r)):.3f} | "
      f"MAE: {mean_absolute_error(y_test_r, my_pred_r):.3f}")
print(f"sklearn   RMSE: {np.sqrt(mean_squared_error(y_test_r, sk_pred_r)):.3f} | "
      f"MAE: {mean_absolute_error(y_test_r, sk_pred_r):.3f}")

MyLinear/MyLogistic НА БЕЙЗЛАЙНЕ (п. 4a–4e)

--- Классификация: MyLogisticRegression vs sklearn (бейзлайн) ---
MyLogReg  Accuracy: 0.505 | ROC-AUC: 0.507 | F1: 0.508
sklearn   Accuracy: 0.505 | ROC-AUC: 0.506 | F1: 0.560

--- Регрессия: MyLinearRegression vs sklearn (бейзлайн) ---
MyLinReg  RMSE: 3.826 | MAE: 2.355
sklearn   RMSE: 3.826 | MAE: 2.355


## 7. MyLinear/MyLogistic с техниками улучшенного бейзлайна (п. 4f–4j)

**Классификация:** one-hot + `C=0.01` (в `MyLogisticRegression` это `alpha=100` — обратная величина).
**Регрессия:** удаление выбросов KDA + `MyLinearRegression`.

Сравниваем с улучшенным sklearn из п. 3.

In [17]:
print("=" * 60)
print("MyLinear/MyLogistic С ТЕХНИКАМИ УЛУЧШЕННОГО БЕЙЗЛАЙНА (п. 4f–4j)")
print("=" * 60)

# --- Классификация: one-hot + C=0.01 ---
ohe_my = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_class_ohe = ohe_my.fit_transform(X_class)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class_ohe, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# sklearn — C=0.01
sk_clf_best = LogisticRegression(C=0.01, max_iter=2000, random_state=42)
sk_clf_best.fit(X_train_c, y_train_c)
sk_pred = sk_clf_best.predict(X_test_c)
sk_proba = sk_clf_best.predict_proba(X_test_c)[:, 1]

# MyLogisticRegression — alpha = 1/C = 100, на подвыборке
# При alpha=100 нужен маленький lr, иначе веса разлетаются
n_sub = min(15000, X_train_c.shape[0])
idx_sub = np.random.RandomState(42).choice(X_train_c.shape[0], n_sub, replace=False)
X_train_sub = X_train_c[idx_sub]
y_train_sub = y_train_c.values[idx_sub]

my_clf_best = MyLogisticRegression(lr=0.01, n_iters=3000, alpha=100.0)
my_clf_best.fit(X_train_sub, y_train_sub)
my_pred = my_clf_best.predict(X_test_c)
my_proba = my_clf_best.predict_proba(X_test_c)

# Проверка на NaN/inf перед метриками
if not np.all(np.isfinite(my_proba)):
    print("ВНИМАНИЕ: MyLogReg дал NaN/inf — уменьшите lr или alpha")
else:
    print("\n--- Классификация: MyLogisticRegression vs sklearn (улучшенный) ---")
    print(f"MyLogReg  Accuracy: {accuracy_score(y_test_c, my_pred):.3f} | "
          f"ROC-AUC: {roc_auc_score(y_test_c, my_proba):.3f} | "
          f"F1: {f1_score(y_test_c, my_pred):.3f}")
    print(f"sklearn   Accuracy: {accuracy_score(y_test_c, sk_pred):.3f} | "
          f"ROC-AUC: {roc_auc_score(y_test_c, sk_proba):.3f} | "
          f"F1: {f1_score(y_test_c, sk_pred):.3f}")

# --- Регрессия: удаление выбросов + MyLinearRegression ---
threshold = y_reg.quantile(0.99)
mask = y_reg <= threshold
X_reg_clean = X_reg[mask]
y_reg_clean = y_reg[mask]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg_clean, y_reg_clean, test_size=0.2, random_state=42
)
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

sk_reg_best = LinearRegression()
sk_reg_best.fit(X_train_r_scaled, y_train_r)
sk_pred_r = sk_reg_best.predict(X_test_r_scaled)

my_reg_best = MyLinearRegression()
my_reg_best.fit(X_train_r_scaled, y_train_r.values)
my_pred_r = my_reg_best.predict(X_test_r_scaled)

print("\n--- Регрессия: MyLinearRegression vs sklearn (улучшенный) ---")
print(f"MyLinReg  RMSE: {np.sqrt(mean_squared_error(y_test_r, my_pred_r)):.3f} | "
      f"MAE: {mean_absolute_error(y_test_r, my_pred_r):.3f}")
print(f"sklearn   RMSE: {np.sqrt(mean_squared_error(y_test_r, sk_pred_r)):.3f} | "
      f"MAE: {mean_absolute_error(y_test_r, sk_pred_r):.3f}")

MyLinear/MyLogistic С ТЕХНИКАМИ УЛУЧШЕННОГО БЕЙЗЛАЙНА (п. 4f–4j)

--- Классификация: MyLogisticRegression vs sklearn (улучшенный) ---
MyLogReg  Accuracy: 0.532 | ROC-AUC: 0.539 | F1: 0.547
sklearn   Accuracy: 0.527 | ROC-AUC: 0.547 | F1: 0.555

--- Регрессия: MyLinearRegression vs sklearn (улучшенный) ---
MyLinReg  RMSE: 3.450 | MAE: 2.084
sklearn   RMSE: 3.450 | MAE: 2.084


## 8. Вывод по лабораторной №2

- Бейзлайн LogReg: Accuracy 0.505, ROC-AUC 0.506, F1 0.560; LinReg: RMSE 3.826, MAE 2.355.
- Улучшенный бейзлайн: Accuracy 0.527, ROC-AUC 0.547, F1 0.555; RMSE 3.450, MAE 2.084.
- `MyLinearRegression` (нормальное уравнение) точно воспроизводит sklearn.
- `MyLogisticRegression` (градиентный спуск) воспроизводит sklearn с точностью до 2–3-го знака — расхождения из-за итеративности и подвыборки.
- Логистическая регрессия лучше KNN в классификации (0.527 vs 0.513 Accuracy), но хуже в регрессии (3.450 vs 3.160 RMSE).